# Financial Portfolio Manager
### Multi-Agent System using AutoGen Group Chat and StateFlow

In [14]:
!pip install autogen groq -q

In [15]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print("GROQ API Key loaded successfully!")

GROQ API Key loaded successfully!


In [16]:
import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import json
import re

# LLM Configuration using Groq
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": GROQ_API_KEY,
            "base_url": "https://api.groq.com/openai/v1",
            "api_type": "openai"
        }
    ],
    "temperature": 0.3,
    "cache_seed": None
}

print("LLM Configuration ready!")

LLM Configuration ready!


In [17]:
# ============================================================
# USER PORTFOLIO DATA
# ============================================================
user_portfolio = {
    "name": "Rahul Sharma",
    "annual_salary": 1200000,  # INR
    "age": 32,
    "risk_appetite": "Moderate",
    "investment_horizon": "7-10 years",
    "current_portfolio": {
        "Fixed Deposits": 300000,
        "SIP (Mutual Funds)": 150000,
        "Real Estate": 2500000,
        "Stocks (Direct Equity)": 200000,
        "Gold": 100000,
        "PPF": 120000,
        "Cash/Savings": 80000
    },
    "monthly_investable_surplus": 25000,
    "financial_goals": ["Retirement", "Child Education", "Wealth Creation"]
}

total_portfolio_value = sum(user_portfolio["current_portfolio"].values())
print(f"User: {user_portfolio['name']}")
print(f"Total Portfolio Value: ₹{total_portfolio_value:,}")
print(f"Annual Salary: ₹{user_portfolio['annual_salary']:,}")

User: Rahul Sharma
Total Portfolio Value: ₹3,450,000
Annual Salary: ₹1,200,000


In [18]:
# ============================================================
# STATEFLOW IMPLEMENTATION
# ============================================================

class StateFlow:
    """
    StateFlow dynamically manages the workflow between agents
    based on portfolio analysis results.
    """

    STATES = {
        "INIT": "Initializing portfolio analysis",
        "PORTFOLIO_ANALYSIS": "Analyzing user portfolio",
        "GROWTH_INVESTMENT": "Generating growth investment recommendations",
        "VALUE_INVESTMENT": "Generating value investment recommendations",
        "ADVISORY": "Compiling personalized financial report",
        "COMPLETED": "Report generated successfully"
    }

    def __init__(self):
        self.current_state = "INIT"
        self.investment_category = None
        self.state_history = ["INIT"]
        print(f"[StateFlow] Initialized → State: {self.current_state}")

    def transition(self, new_state):
        """Transition to a new state."""
        if new_state in self.STATES:
            print(f"[StateFlow] {self.current_state} → {new_state}: {self.STATES[new_state]}")
            self.current_state = new_state
            self.state_history.append(new_state)
        else:
            raise ValueError(f"Invalid state: {new_state}")

    def determine_investment_type(self, portfolio_data):
        """
        Determines whether to pursue Growth or Value investment
        based on portfolio composition and user profile.
        """
        total = sum(portfolio_data["current_portfolio"].values())

        # Calculate equity exposure percentage
        equity = portfolio_data["current_portfolio"].get("Stocks (Direct Equity)", 0)
        sip = portfolio_data["current_portfolio"].get("SIP (Mutual Funds)", 0)
        equity_pct = (equity + sip) / total * 100

        # Calculate fixed income exposure
        fixed = portfolio_data["current_portfolio"].get("Fixed Deposits", 0)
        ppf = portfolio_data["current_portfolio"].get("PPF", 0)
        fixed_pct = (fixed + ppf) / total * 100

        age = portfolio_data["age"]
        risk = portfolio_data["risk_appetite"]

        # StateFlow Decision Logic
        print(f"\n[StateFlow Analysis]")
        print(f"  Equity Exposure: {equity_pct:.1f}%")
        print(f"  Fixed Income Exposure: {fixed_pct:.1f}%")
        print(f"  Age: {age} | Risk Appetite: {risk}")

        if age < 40 and risk in ["Moderate", "High"] and equity_pct < 40:
            self.investment_category = "GROWTH"
            self.transition("GROWTH_INVESTMENT")
            print(f"  Decision: GROWTH Investment (under-allocated to equity for age/risk profile)")
        elif fixed_pct > 30 or age >= 40 or risk == "Low":
            self.investment_category = "VALUE"
            self.transition("VALUE_INVESTMENT")
            print(f"  Decision: VALUE Investment (heavy fixed income or conservative profile)")
        else:
            self.investment_category = "GROWTH"
            self.transition("GROWTH_INVESTMENT")
            print(f"  Decision: GROWTH Investment (default for balanced young investor)")

        return self.investment_category

    def get_state_summary(self):
        return {
            "current_state": self.current_state,
            "investment_category": self.investment_category,
            "state_history": self.state_history
        }

# Initialize StateFlow
stateflow = StateFlow()
print("\nStateFlow engine initialized!")

[StateFlow] Initialized → State: INIT

StateFlow engine initialized!


In [19]:
# ============================================================
# AGENT DEFINITIONS
# ============================================================

# Agent 1: User Proxy Agent
user_proxy = UserProxyAgent(
    name="UserProxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    system_message="""You are the User Proxy representing the investor.
    You initiate the conversation and provide the user's portfolio data to the group.
    You do not generate responses - you only initiate the task."""
)

# Agent 2: Portfolio Analysis Agent
portfolio_analysis_agent = AssistantAgent(
    name="PortfolioAnalysisAgent",
    llm_config=llm_config,
    system_message="""You are an expert Portfolio Analysis Agent.

    Your role:
    1. Analyze the user's current investment portfolio in detail
    2. Calculate portfolio allocation percentages across asset classes
    3. Identify gaps, over-allocations, and under-allocations
    4. Assess the portfolio against the user's age, salary, risk appetite, and goals
    5. Determine whether the user should pursue GROWTH or VALUE investments
    6. Provide a clear summary with key observations

    Structure your response as:
    ## Portfolio Analysis Summary
    - Current Holdings Breakdown (with % allocation)
    - Key Observations
    - Portfolio Health Assessment
    - Investment Category Recommendation: [GROWTH/VALUE]
    - Rationale for recommendation

    End your response with: INVESTMENT_CATEGORY: [GROWTH or VALUE]"""
)

# Agent 3: Growth Investment Agent
growth_investment_agent = AssistantAgent(
    name="GrowthInvestmentAgent",
    llm_config=llm_config,
    system_message="""You are a Growth Investment Specialist.

    Your role:
    You are activated when a user needs GROWTH investments to maximize portfolio returns.

    Provide recommendations for:
    1. High-growth equity mutual funds (specify fund categories and examples)
    2. Direct stock investment sectors (with growth rationale)
    3. SIP recommendations (amounts and fund types)
    4. Emerging market or sectoral opportunities
    5. Technology/innovation investment options
    6. Suggested portfolio rebalancing for growth

    Structure your response as:
    ## Growth Investment Recommendations
    - Top Mutual Fund Categories & Examples
    - Direct Equity Sectors to Target
    - SIP Strategy
    - Alternative Growth Investments
    - Suggested Monthly Allocation Plan
    - Expected Returns (realistic range)
    - Risk Mitigation Tips"""
)

# Agent 4: Value Investment Agent
value_investment_agent = AssistantAgent(
    name="ValueInvestmentAgent",
    llm_config=llm_config,
    system_message="""You are a Value Investment Specialist.

    Your role:
    You are activated when a user needs VALUE investments for stable long-term wealth creation.

    Provide recommendations for:
    1. Blue-chip dividend stocks and value stocks
    2. Debt mutual funds and hybrid funds
    3. Government securities and bonds
    4. Real estate investment trusts (REITs)
    5. Gold and commodity allocation
    6. Tax-efficient investment vehicles (ELSS, PPF, NPS)

    Structure your response as:
    ## Value Investment Recommendations
    - Blue-Chip Stocks & Dividend Plays
    - Debt & Hybrid Mutual Funds
    - Government Instruments
    - Alternative Value Investments
    - Suggested Monthly Allocation Plan
    - Expected Returns (realistic range)
    - Capital Preservation Strategy"""
)

# Agent 5: Investment Advisor Agent
investment_advisor_agent = AssistantAgent(
    name="InvestmentAdvisorAgent",
    llm_config=llm_config,
    system_message="""You are a Senior Investment Advisor who compiles personalized financial reports.

    Your role:
    Synthesize all insights from the Portfolio Analysis Agent and the Investment Recommendations
    (either Growth or Value) into a comprehensive, personalized financial report.

    Your report MUST include:

    ## PERSONALIZED FINANCIAL REPORT
    ### Executive Summary
    ### Current Portfolio Assessment
    ### Investment Strategy (Growth/Value)
    ### Detailed Action Plan
       - Immediate actions (0-3 months)
       - Short-term actions (3-12 months)
       - Long-term strategy (1-5 years)
    ### Recommended Portfolio Allocation
    ### Monthly Investment Plan
    ### Goal-wise Investment Mapping
    ### Tax Planning Considerations
    ### Risk Assessment & Mitigation
    ### Key Metrics & Targets
    ### Disclaimer

    Make the report professional, specific, and actionable.
    Address the user by name and personalize all recommendations based on their profile."""
)

print("All agents initialized successfully!")
print(f"Agents: UserProxy, PortfolioAnalysisAgent, GrowthInvestmentAgent, ValueInvestmentAgent, InvestmentAdvisorAgent")

All agents initialized successfully!
Agents: UserProxy, PortfolioAnalysisAgent, GrowthInvestmentAgent, ValueInvestmentAgent, InvestmentAdvisorAgent


In [20]:
# ============================================================
# STEP 1 & 2: PORTFOLIO ANALYSIS VIA GROUP CHAT
# ============================================================

print("=" * 70)
print("STEP 1 & 2: Initiating Portfolio Analysis Group Chat")
print("=" * 70)

stateflow.transition("PORTFOLIO_ANALYSIS")

# Group Chat for Portfolio Analysis
analysis_group_chat = GroupChat(
    agents=[user_proxy, portfolio_analysis_agent],
    messages=[],
    max_round=3,
    speaker_selection_method="round_robin"
)

analysis_manager = GroupChatManager(
    groupchat=analysis_group_chat,
    llm_config=llm_config
)

# Initiate portfolio analysis
portfolio_message = f"""
Please analyze the following investment portfolio and provide a comprehensive analysis:

INVESTOR PROFILE:
- Name: {user_portfolio['name']}
- Age: {user_portfolio['age']} years
- Annual Salary: ₹{user_portfolio['annual_salary']:,}
- Risk Appetite: {user_portfolio['risk_appetite']}
- Investment Horizon: {user_portfolio['investment_horizon']}
- Monthly Investable Surplus: ₹{user_portfolio['monthly_investable_surplus']:,}
- Financial Goals: {', '.join(user_portfolio['financial_goals'])}

CURRENT PORTFOLIO:
{json.dumps(user_portfolio['current_portfolio'], indent=2)}

Total Portfolio Value: ₹{total_portfolio_value:,}

Please analyze this portfolio and determine whether GROWTH or VALUE investment strategy is recommended.
"""

print("\n[UserProxy → GroupChatManager → PortfolioAnalysisAgent]\n")

user_proxy.initiate_chat(
    analysis_manager,
    message=portfolio_message
)

# Extract portfolio analysis result
analysis_messages = analysis_group_chat.messages
portfolio_analysis_result = ""
for msg in analysis_messages:
    if msg.get('name') == 'PortfolioAnalysisAgent':
        portfolio_analysis_result = msg.get('content', '')
        break

print("\n" + "=" * 70)
print("Portfolio Analysis Complete!")
print("=" * 70)

STEP 1 & 2: Initiating Portfolio Analysis Group Chat
[StateFlow] INIT → PORTFOLIO_ANALYSIS: Analyzing user portfolio

[UserProxy → GroupChatManager → PortfolioAnalysisAgent]

UserProxy (to chat_manager):


Please analyze the following investment portfolio and provide a comprehensive analysis:

INVESTOR PROFILE:
- Name: Rahul Sharma
- Age: 32 years
- Annual Salary: ₹1,200,000
- Risk Appetite: Moderate
- Investment Horizon: 7-10 years
- Monthly Investable Surplus: ₹25,000
- Financial Goals: Retirement, Child Education, Wealth Creation

CURRENT PORTFOLIO:
{
  "Fixed Deposits": 300000,
  "SIP (Mutual Funds)": 150000,
  "Real Estate": 2500000,
  "Stocks (Direct Equity)": 200000,
  "Gold": 100000,
  "PPF": 120000,
  "Cash/Savings": 80000
}

Total Portfolio Value: ₹3,450,000

Please analyze this portfolio and determine whether GROWTH or VALUE investment strategy is recommended.


--------------------------------------------------------------------------------

Next speaker: PortfolioAnalysisA

PortfolioAnalysisAgent (to chat_manager):

## Portfolio Analysis Summary
- Current Holdings Breakdown (with % allocation):
  - Fixed Deposits: ₹300,000 (8.7%)
  - SIP (Mutual Funds): ₹150,000 (4.3%)
  - Real Estate: ₹2,500,000 (72.5%)
  - Stocks (Direct Equity): ₹200,000 (5.8%)
  - Gold: ₹100,000 (2.9%)
  - PPF: ₹120,000 (3.5%)
  - Cash/Savings: ₹80,000 (2.3%)
- Key Observations:
  - The portfolio is heavily skewed towards Real Estate, which accounts for approximately 72.5% of the total portfolio value.
  - The allocation to Stocks (Direct Equity) and SIP (Mutual Funds) is relatively low, which may not be sufficient to achieve long-term growth goals.
  - The portfolio has a moderate allocation to low-risk investments such as Fixed Deposits, PPF, and Cash/Savings.
  - The investor's risk appetite is moderate, but the portfolio's overall risk profile is relatively high due to the significant exposure to Real Estate.
- Portfolio Health Assessment:
  - The portfolio is not well-diversified

In [21]:
# ============================================================
# STEP 3: STATEFLOW - DETERMINE INVESTMENT CATEGORY
# ============================================================

print("=" * 70)
print("STEP 3: StateFlow - Dynamic Workflow Management")
print("=" * 70)

# Use StateFlow to determine investment type
investment_category = stateflow.determine_investment_type(user_portfolio)

# Also check the LLM's recommendation from portfolio analysis
if "INVESTMENT_CATEGORY: GROWTH" in portfolio_analysis_result.upper():
    llm_recommendation = "GROWTH"
elif "INVESTMENT_CATEGORY: VALUE" in portfolio_analysis_result.upper():
    llm_recommendation = "VALUE"
else:
    llm_recommendation = investment_category  # Fall back to StateFlow decision

print(f"\n[StateFlow Decision]: {investment_category}")
print(f"[LLM Recommendation]: {llm_recommendation}")

# Final decision (LLM recommendation takes priority if available)
final_category = llm_recommendation
print(f"[Final Category]: {final_category} Investment Strategy Selected")

print("\n" + "=" * 70)
print(f"StateFlow routing workflow to: {final_category} Investment Agent")
print("=" * 70)

STEP 3: StateFlow - Dynamic Workflow Management

[StateFlow Analysis]
  Equity Exposure: 10.1%
  Fixed Income Exposure: 12.2%
  Age: 32 | Risk Appetite: Moderate
[StateFlow] PORTFOLIO_ANALYSIS → GROWTH_INVESTMENT: Generating growth investment recommendations
  Decision: GROWTH Investment (under-allocated to equity for age/risk profile)

[StateFlow Decision]: GROWTH
[LLM Recommendation]: GROWTH
[Final Category]: GROWTH Investment Strategy Selected

StateFlow routing workflow to: GROWTH Investment Agent


In [22]:
# ============================================================
# STEP 4: INVESTMENT RECOMMENDATIONS (GROWTH OR VALUE)
# ============================================================

print("=" * 70)
print(f"STEP 4: {final_category} Investment Agent Generating Recommendations")
print("=" * 70)

# Select the appropriate investment agent based on StateFlow decision
if final_category == "GROWTH":
    selected_investment_agent = growth_investment_agent
    stateflow.current_state = "GROWTH_INVESTMENT"
else:
    selected_investment_agent = value_investment_agent
    stateflow.current_state = "VALUE_INVESTMENT"

print(f"\n[StateFlow] Activated: {selected_investment_agent.name}")

# Group Chat for Investment Recommendations
investment_group_chat = GroupChat(
    agents=[user_proxy, selected_investment_agent],
    messages=[],
    max_round=3,
    speaker_selection_method="round_robin"
)

investment_manager = GroupChatManager(
    groupchat=investment_group_chat,
    llm_config=llm_config
)

investment_request = f"""
Based on the portfolio analysis for {user_portfolio['name']}, the StateFlow system has routed
this case to the {final_category} Investment strategy.

PORTFOLIO ANALYSIS SUMMARY:
{portfolio_analysis_result[:1500]}...

INVESTOR PROFILE:
- Age: {user_portfolio['age']} | Salary: ₹{user_portfolio['annual_salary']:,}/year
- Risk Appetite: {user_portfolio['risk_appetite']}
- Monthly Surplus for Investment: ₹{user_portfolio['monthly_investable_surplus']:,}
- Investment Horizon: {user_portfolio['investment_horizon']}
- Goals: {', '.join(user_portfolio['financial_goals'])}

Please provide detailed {final_category} investment recommendations tailored to this investor's profile.
"""

print(f"\n[GroupChat] UserProxy → GroupChatManager → {selected_investment_agent.name}\n")

user_proxy.initiate_chat(
    investment_manager,
    message=investment_request
)

# Extract investment recommendations
investment_messages = investment_group_chat.messages
investment_recommendations = ""
for msg in investment_messages:
    if msg.get('name') in ['GrowthInvestmentAgent', 'ValueInvestmentAgent']:
        investment_recommendations = msg.get('content', '')
        break

print("\n" + "=" * 70)
print("Investment Recommendations Generated!")
print("=" * 70)

STEP 4: GROWTH Investment Agent Generating Recommendations

[StateFlow] Activated: GrowthInvestmentAgent

[GroupChat] UserProxy → GroupChatManager → GrowthInvestmentAgent

UserProxy (to chat_manager):


Based on the portfolio analysis for Rahul Sharma, the StateFlow system has routed
this case to the GROWTH Investment strategy.

PORTFOLIO ANALYSIS SUMMARY:
## Portfolio Analysis Summary
- Current Holdings Breakdown (with % allocation):
  - Fixed Deposits: ₹300,000 (8.7%)
  - SIP (Mutual Funds): ₹150,000 (4.3%)
  - Real Estate: ₹2,500,000 (72.5%)
  - Stocks (Direct Equity): ₹200,000 (5.8%)
  - Gold: ₹100,000 (2.9%)
  - PPF: ₹120,000 (3.5%)
  - Cash/Savings: ₹80,000 (2.3%)
- Key Observations:
  - The portfolio is heavily skewed towards Real Estate, which accounts for approximately 72.5% of the total portfolio value.
  - The allocation to Stocks (Direct Equity) and SIP (Mutual Funds) is relatively low, which may not be sufficient to achieve long-term growth goals.
  - The portfolio has a m

GrowthInvestmentAgent (to chat_manager):

## Growth Investment Recommendations
Based on Rahul Sharma's portfolio analysis and investor profile, I provide the following growth investment recommendations to maximize portfolio returns:

### Top Mutual Fund Categories & Examples
To diversify the portfolio and increase exposure to growth-oriented assets, I recommend investing in the following mutual fund categories:
- **Mid-Cap Funds**: Invest ₹5,000/month in mid-cap funds like Franklin India Feeder - Franklin U.S. Opportunities Fund or UTI NIFTY INDEX FUND.
- **Small-Cap Funds**: Invest ₹5,000/month in small-cap funds like SBI MAGNUM MULTICAP FUND or HDFC SMALL CAP FUND.
- **Sectoral Funds (Technology)**: Invest ₹5,000/month in sectoral funds like FRANKLIN INDIA FLEXICAP FUND or ICICI PRUDENTIAL TECHNOLOGY FUND.

### Direct Equity Sectors to Target
To further diversify the portfolio and tap into growth potential, I recommend investing in the following direct equity sectors:
- **Technology*

In [23]:
# ============================================================
# STEP 5: INVESTMENT ADVISOR - FINAL REPORT
# ============================================================

print("=" * 70)
print("STEP 5: Investment Advisor Agent - Generating Personalized Report")
print("=" * 70)

stateflow.transition("ADVISORY")

# Group Chat for Final Report
advisory_group_chat = GroupChat(
    agents=[user_proxy, investment_advisor_agent],
    messages=[],
    max_round=3,
    speaker_selection_method="round_robin"
)

advisory_manager = GroupChatManager(
    groupchat=advisory_group_chat,
    llm_config=llm_config
)

advisory_request = f"""
Please compile a comprehensive, personalized financial report for {user_portfolio['name']}
using the following insights from our analysis agents:

══════════════════════════════════════════════════════
PORTFOLIO ANALYSIS (from PortfolioAnalysisAgent):
══════════════════════════════════════════════════════
{portfolio_analysis_result}

══════════════════════════════════════════════════════
{final_category} INVESTMENT RECOMMENDATIONS (from {selected_investment_agent.name}):
══════════════════════════════════════════════════════
{investment_recommendations}

══════════════════════════════════════════════════════
COMPLETE INVESTOR PROFILE:
══════════════════════════════════════════════════════
Name: {user_portfolio['name']}
Age: {user_portfolio['age']}
Annual Salary: ₹{user_portfolio['annual_salary']:,}
Total Portfolio Value: ₹{total_portfolio_value:,}
Monthly Investable Surplus: ₹{user_portfolio['monthly_investable_surplus']:,}
Risk Appetite: {user_portfolio['risk_appetite']}
Investment Horizon: {user_portfolio['investment_horizon']}
Financial Goals: {', '.join(user_portfolio['financial_goals'])}
Investment Strategy Selected by StateFlow: {final_category}

Please generate the complete personalized financial report.
"""

print("\n[GroupChat] All Agents → InvestmentAdvisorAgent → Final Report\n")

user_proxy.initiate_chat(
    advisory_manager,
    message=advisory_request
)

# Extract final report
advisory_messages = advisory_group_chat.messages
final_report = ""
for msg in advisory_messages:
    if msg.get('name') == 'InvestmentAdvisorAgent':
        final_report = msg.get('content', '')
        break

stateflow.transition("COMPLETED")

print("\n" + "=" * 70)
print("Personalized Financial Report Generated!")
print("=" * 70)

STEP 5: Investment Advisor Agent - Generating Personalized Report
[StateFlow] GROWTH_INVESTMENT → ADVISORY: Compiling personalized financial report

[GroupChat] All Agents → InvestmentAdvisorAgent → Final Report

UserProxy (to chat_manager):


Please compile a comprehensive, personalized financial report for Rahul Sharma 
using the following insights from our analysis agents:

══════════════════════════════════════════════════════
PORTFOLIO ANALYSIS (from PortfolioAnalysisAgent):
══════════════════════════════════════════════════════
## Portfolio Analysis Summary
- Current Holdings Breakdown (with % allocation):
  - Fixed Deposits: ₹300,000 (8.7%)
  - SIP (Mutual Funds): ₹150,000 (4.3%)
  - Real Estate: ₹2,500,000 (72.5%)
  - Stocks (Direct Equity): ₹200,000 (5.8%)
  - Gold: ₹100,000 (2.9%)
  - PPF: ₹120,000 (3.5%)
  - Cash/Savings: ₹80,000 (2.3%)
- Key Observations:
  - The portfolio is heavily skewed towards Real Estate, which accounts for approximately 72.5% of the total portfolio v

InvestmentAdvisorAgent (to chat_manager):

## PERSONALIZED FINANCIAL REPORT

### Executive Summary
Dear Rahul Sharma, this personalized financial report is designed to provide you with a comprehensive overview of your current financial situation, investment strategy, and recommendations for achieving your long-term goals. Based on our analysis, we have identified areas of improvement in your portfolio and suggest a growth-oriented investment strategy to maximize returns. Our report outlines a detailed action plan, recommended portfolio allocation, and monthly investment plan to help you achieve your objectives.

### Current Portfolio Assessment
Your current portfolio is valued at ₹3,450,000, with a significant allocation to Real Estate (72.5%). While this provides a stable source of income, it may not be sufficient to achieve your long-term growth goals. The allocation to Stocks (Direct Equity) and SIP (Mutual Funds) is relatively low, which may limit the portfolio's return potential. 

In [24]:
# ============================================================
# FINAL OUTPUT: DISPLAY THE PERSONALIZED FINANCIAL REPORT
# ============================================================

print("\n" + "█" * 70)
print("█" + " " * 18 + "PERSONALIZED FINANCIAL REPORT" + " " * 21 + "█")
print("█" + " " * 68 + "█")
print("█" + f"  Generated by: Multi-Agent Financial Portfolio Manager System" + " " * 7 + "█")
print("█" + f"  Investment Strategy: {final_category} Investment (StateFlow Decision)" + " " * (27 - len(final_category)) + "█")
print("█" * 70)
print()
print(final_report)
print()
print("═" * 70)
print("STATEFLOW WORKFLOW SUMMARY")
print("═" * 70)
state_summary = stateflow.get_state_summary()
print(f"States Traversed: {' → '.join(state_summary['state_history'])}")
print(f"Final State: {state_summary['current_state']}")
print(f"Investment Category Determined: {state_summary['investment_category']}")
print("═" * 70)
print("\n✅ Financial Portfolio Manager - Task Completed Successfully!")


██████████████████████████████████████████████████████████████████████
█                  PERSONALIZED FINANCIAL REPORT                     █
█                                                                    █
█  Generated by: Multi-Agent Financial Portfolio Manager System       █
█  Investment Strategy: GROWTH Investment (StateFlow Decision)                     █
██████████████████████████████████████████████████████████████████████

## PERSONALIZED FINANCIAL REPORT

### Executive Summary
Dear Rahul Sharma, this personalized financial report is designed to provide you with a comprehensive overview of your current financial situation, investment strategy, and recommendations for achieving your long-term goals. Based on our analysis, we have identified areas of improvement in your portfolio and suggest a growth-oriented investment strategy to maximize returns. Our report outlines a detailed action plan, recommended portfolio allocation, and monthly investment plan to help you achiev

In [25]:
# ============================================================
# AGENT CONVERSATION FLOW VISUALIZATION
# ============================================================

print("\n" + "=" * 70)
print("AGENT COLLABORATION FLOW")
print("=" * 70)

flow_diagram = """
┌─────────────────────────────────────────────────────────────┐
│              FINANCIAL PORTFOLIO MANAGER FLOW               │
└─────────────────────────────────────────────────────────────┘

  [UserProxy Agent]
       │
       ▼ initiates chat
  [GroupChat Manager]
       │
       ▼ routes to
  [Portfolio Analysis Agent]
       │  • Analyzes current holdings
       │  • Calculates allocation %
       │  • Assesses risk profile
       │
       ▼ outputs recommendation
  ┌─────────────────────┐
  │   StateFlow Engine  │
  │  Dynamic Routing    │
  └─────────────────────┘
       │
    ┌──┴──┐
    ▼     ▼
[GROWTH] [VALUE]
Investment Investment
 Agent    Agent
    │     │
    └──┬──┘
       ▼ recommendations forwarded
  [Investment Advisor Agent]
       │  • Compiles all insights
       │  • Creates personalized report
       │  • Maps goals to investments
       │
       ▼
  [PERSONALIZED FINANCIAL REPORT]
"""

print(flow_diagram)
print(f"Investment Strategy Selected: {final_category}")
print(f"Agent Activated: {selected_investment_agent.name}")


AGENT COLLABORATION FLOW

┌─────────────────────────────────────────────────────────────┐
│              FINANCIAL PORTFOLIO MANAGER FLOW               │
└─────────────────────────────────────────────────────────────┘

  [UserProxy Agent]
       │
       ▼ initiates chat
  [GroupChat Manager]
       │
       ▼ routes to
  [Portfolio Analysis Agent]
       │  • Analyzes current holdings
       │  • Calculates allocation %
       │  • Assesses risk profile
       │
       ▼ outputs recommendation
  ┌─────────────────────┐
  │   StateFlow Engine  │
  │  Dynamic Routing    │
  └─────────────────────┘
       │
    ┌──┴──┐
    ▼     ▼
[GROWTH] [VALUE]
Investment Investment
 Agent    Agent
    │     │
    └──┬──┘
       ▼ recommendations forwarded
  [Investment Advisor Agent]
       │  • Compiles all insights
       │  • Creates personalized report
       │  • Maps goals to investments
       │
       ▼
  [PERSONALIZED FINANCIAL REPORT]

Investment Strategy Selected: GROWTH
Agent Activated: 